# Session6. Сабмиты после полного переобучения

Продолжение эксперимента `clustering326`. На локальном test архивный Session6 дал 0.319190, новая общая LightGBM при его объёме дала 0.319163, LightGBM по кластерам дала 0.319496. Общая LightGBM также выиграла на val. Эти оценки получены до полного переобучения, в предыдущем запуске. Локальный период ранее изучался.

В Kaggle оставить подключённые датасеты `case_rwb`, `cleaning_inputs` и `s6_residual_inputs`, включить GPU и выполнить Run All. `case_rwb` даёт train/test parquet, `cleaning_inputs` — параметры LightGBM, `s6_residual_inputs` — OOF-прогнозы CatBoost и признаки для поправки. Файлы можно хранить распакованными или внутри ZIP. Если `case_rwb` отсутствует, train/test читаются из `cleaning_inputs`.

Восемь CatBoost заново обучаются на всех доступных получасовых стартах с известной целью, включая бывшие val и локальный test. Это расширяет окно по сравнению с recent75 в исходном архивном CatBoost. LightGBM обучается на пяти утренних стартах за всю доступную размеченную историю с нужным прогревом признаков. KMeans, масштабы маршрутов и корреляционные лаги используют историю до конкурсного cutoff. Две модели поправки CatBoost переобучаются на всех доступных исходных OOF за 4–31 октября. Остатки этой истории дают причинный EWMA-сдвиг на конкурсный cutoff. CatBoost обучается на GPU, LightGBM на CPU.

`submission.csv` использует общую LightGBM, `submission_clusters.csv` использует восемь кластеров с общей моделью для малых групп. Обе формы нормируются к объёму заново обученного компонента Session6 с исходным весом 0.4997259. Обе сохраняют одинаковые суммы горизонтов. После каждого обучения выводятся метрики на обучении полного refit, а архивная независимая оценка показана отдельной таблицей. Финальный ZIP содержит модели, метрики, прогнозы и оба CSV. Новый результат на платформе станет известен после отправки файла.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import gc
import json
import zipfile

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
search = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd() / 'best'
RUN = work / 'session6_full_submissions'
RUN.mkdir(exist_ok=True)
fit_end = pd.Timestamp('2025-10-04')
test_start = pd.Timestamp('2025-10-18')
test_end = pd.Timestamp('2025-11-01')
STEP = pd.Timedelta(minutes=30)
CLUSTERS = (1, 8)
MIN_CLUSTER_ROUTES = 10
SEED = 42
TARGET = 'target_1h'
names = [TARGET, *[f'status_{i}' for i in range(1, 7)]]
keys = ['cutoff', 'horizon', 'route_id']
group_keys = ['cutoff', 'horizon']
from shutil import copyfileobj


def unique_path(paths, label):
    if len(paths) != 1:
        raise ValueError(f'{label}: ожидался один путь, найдено {len(paths)}: {paths}')
    return paths[0]


def dataset_mount(slug):
    normalized = slug.replace('_', '-').lower()
    matches = [path for path in search.rglob('*') if path.is_dir() and path.name.replace('_', '-').lower() == normalized]
    return unique_path(matches, slug) if matches else None


def dataset_files(slug, filenames, archive_name=None, mount=None):
    mount = mount if mount is not None else dataset_mount(slug)
    resolved = {}
    if mount is not None:
        for filename in filenames:
            matches = [path for path in mount.rglob(filename) if path.is_file()]
            if matches:
                resolved[filename] = unique_path(matches, f'{slug}/{filename}')
                print(f'{slug}/{filename} <- {resolved[filename]}')
    missing = [filename for filename in filenames if filename not in resolved]
    if not missing:
        return resolved
    if archive_name is None:
        raise FileNotFoundError(f'В датасете {slug} отсутствуют файлы {missing}')
    archives = list((mount or search).rglob(archive_name))
    if not archives and mount is not None:
        archives = list(mount.rglob('*.zip'))
    if not archives:
        raise FileNotFoundError(f'Подключите датасет {slug} с файлами {missing} или архивом {archive_name}')
    archive_path = unique_path(archives, f'ZIP датасета {slug}')
    destination = Path(temporary.name) / slug
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for filename in missing:
            members = [item for item in archive.infolist() if not item.is_dir() and Path(item.filename).name == filename]
            member = unique_path(members, f'{archive_path.name}/{filename}')
            target_path = destination / filename
            with archive.open(member) as source, target_path.open('wb') as output:
                copyfileobj(source, output)
            resolved[filename] = target_path
            print(f'{slug}/{filename} <- {archive_path}/{member.filename}')
    return resolved


temporary = TemporaryDirectory()
data_names = ('train_solo_track.parquet', 'test_solo_track.parquet')
case_mount = dataset_mount('case_rwb')
cleaning_names = ('comparison_meta.json',) if case_mount is not None else ('comparison_meta.json', *data_names)
cleaning_inputs = dataset_files('cleaning_inputs', cleaning_names, 'cleaning_comparison_inputs.zip')
case_inputs = dataset_files('case_rwb', data_names, mount=case_mount) if case_mount is not None else cleaning_inputs
RESIDUAL_INPUTS = dataset_files('s6_residual_inputs', ('selected_evaluation.parquet', 'anchor_features.parquet'),
                                'session6_residual_oof_inputs.zip')
TRAIN_INPUT, TEST_INPUT = (case_inputs[filename] for filename in data_names)
meta = json.loads(cleaning_inputs['comparison_meta.json'].read_text())
history = pd.read_parquet(TRAIN_INPUT, columns=['timestamp', 'route_id', *names])
history = history.sort_values(['timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if len(history) != len(times) * len(routes) or history.duplicated(['timestamp', 'route_id']).any() or history.isna().any().any():
    raise ValueError('Неполная панель')
if not times.equals(pd.date_range(times[0], times[-1], freq='30min')):
    raise ValueError('Неверная сетка времени')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in names}
del history
target = values[TARGET]
TIMES, ROUTES, VALUES = times, routes, values

from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count

STATUSES = names[1:]
REFERENCE_SCORES = [
    dict(variant='session_prediction', WAPE=.3178266755292927, relative_bias=-.0013635244235125578, score=.3191901999528053),
    dict(variant='session_global_shape', WAPE=.3177996914357809, relative_bias=-.0013635244235125508, score=.31916321585929347),
    dict(variant='session_clustered_shape', WAPE=.3181329194743572, relative_bias=-.001363524423512551, score=.31949644389786974),
]


In [ ]:
def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f'target_back_{k}'] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f'{name}_mean_{steps}'] = means[name, steps][origins]
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            if name in STATUSES[:3]:
                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)
    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)
    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)
    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])
    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка признаков не сопоставлена с историей')
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))
        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)
    return frame

In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}
    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)
    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    for days in (1, 2, 7):
        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()
    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric})

def add_pilot_features(frame, h, origins):
    current = target[origins]
    level = level24[origins]
    frame['level24_feature'] = level.ravel()
    frame['target_level24_logratio'] = (np.log1p(current) - np.log1p(level)).ravel()
    frame['mean2h_level24_logratio'] = (np.log1p(means[TARGET, 4][origins]) - np.log1p(level)).ravel()
    frame['profile28_minus_daily7'] = (frame.profile28 - frame.slot_daily7).astype(np.float32)
    return frame.astype({c: 'float32' for c in [*SEASONAL_FEATURES, *STATUS_FEATURES]})
SEASONAL_FEATURES = ['profile28', 'level24_feature', 'target_level24_logratio', 'mean2h_level24_logratio', 'profile28_minus_daily7']
STATUS_FEATURES = []

In [ ]:
def extend_robust(data):
    new = {}
    floor = 1 / np.maximum(data.scale.to_numpy(dtype=float), 1)
    for j in range(1, 7):
        name = f'status_{j}'
        level = np.maximum(data[f'long_{name}_mean48'].to_numpy(dtype=float), floor)
        week = np.maximum(data[f'long_{name}_mean336'].to_numpy(dtype=float), floor)
        for suffix, key in [('current', f'{name}_lag0'), ('mean4', f'{name}_mean4'), ('mean12', f'{name}_mean12'), ('delta4', f'{name}_delta4')]:
            new[f'robust_s{j}_{suffix}'] = np.clip(data[key].to_numpy() / level, -100, 100).astype(np.float32)
        new[f'robust_s{j}_level_change'] = np.clip(level / week, 0, 100).astype(np.float32)
        raw = data[f'{name}_lag0'].to_numpy(dtype=float) * data.scale.to_numpy()
        denominator = pd.Series(raw).groupby([data.cutoff, data.horizon], sort=False).transform('sum').to_numpy()
        new[f'robust_s{j}_route_share'] = np.divide(raw, np.maximum(denominator, 1)).astype(np.float32)
    for window in [4, 12]:
        new[f'robust_target_vs{window}'] = np.clip(data.target_lag_0.to_numpy() / np.maximum(data[f'target_mean_{window}'].to_numpy(), floor), 0, 100).astype(np.float32)
    return pd.concat([data, pd.DataFrame(new, index=data.index)], axis=1)

In [ ]:
def session_shape_features(origins, h, profile_history, daily):
    T = len(times)
    R = len(routes)

    def past(a, index):
        index = np.asarray(index)
        out = a[np.clip(index, 0, T - 1)].astype(np.float32, copy=True)
        out[(index < 0) | (index >= T)] = np.nan
        return out
    scale = np.maximum(daily[origins], 1)
    common = {}
    for lag in (*range(9), 12, 24, 48, 96, 336):
        common[f'target_lag_{lag}'] = past(target, origins - lag) / scale
    for window in (4, 12, 48):
        common[f'target_mean_{window}'] = (daily if window == 48 else means['target_1h', window])[origins] / scale
    for window in (4, 12):
        common[f'target_std_{window}'] = deviations[window][origins] / scale
    for name in names[1:]:
        a = values[name]
        for lag in (0, 1, 2):
            common[f'{name}_lag{lag}'] = past(a, origins - lag) / scale
        for window in (4, 12):
            common[f'{name}_mean{window}'] = means[name, window][origins] / scale
        for lag in (2, 4):
            common[f'{name}_delta{lag}'] = (a[origins] - past(a, origins - lag)) / scale
    common['log_current_level'] = np.log1p(scale)
    for name, a in values.items():
        total = a.sum(axis=1, dtype=np.float64)
        netlevel = pd.Series(total).rolling(48, min_periods=1).mean().to_numpy()
        net12 = pd.Series(total).rolling(12, min_periods=1).mean().to_numpy()
        den = np.maximum(netlevel[origins], 1)
        for suffix, x in [('level', total[origins] / den), ('mean12', net12[origins] / den), ('delta2', (total[origins] - total[origins - 2]) / den)]:
            common[f'network_{name}_{suffix}'] = np.broadcast_to(x[:, None], scale.shape).astype(np.float32)
    common['route_network_share'] = target[origins] / np.maximum(target[origins].sum(axis=1, keepdims=True), 1)
    slot = origins + h
    dates = times[origins] + pd.Timedelta(minutes=30 * h)
    arrays = {name: a.ravel() for name, a in common.items()}
    for day in (1, 2, 3, 7, 14, 21, 28):
        a = past(target, slot - day * 48)
        arrays[f'slot_day{day}'] = (a / scale).ravel()
    for day in (1, 7):
        for offset in (-2, -1, 1, 2):
            arrays[f'slot_day{day}_offset{offset}'] = (past(target, slot - day * 48 + offset) / scale).ravel()
    stack = np.stack([past(target, slot - day * 48) for day in range(1, 8)])
    arrays['slot_daily7'] = (np.nanmean(stack, axis=0) / scale).ravel()
    stack = np.stack([past(target, slot - day * 48) for day in (7, 14, 21, 28)])
    with np.errstate(invalid='ignore'):
        weekly = np.nanmean(stack, axis=0)
    arrays['slot_weekly4'] = (weekly / scale).ravel()
    profile_sum = np.zeros_like(scale, dtype=np.float64)
    count = np.zeros_like(scale)
    for day in range(1, 29):
        a = past(profile_history, slot - day * 48) * scale / np.maximum(past(daily, origins - day * 48), 1)
        valid = np.isfinite(a)
        profile_sum += np.nan_to_num(a)
        count += valid
    profile = np.divide(profile_sum, count, out=np.zeros_like(profile_sum), where=count > 0).astype(np.float32)
    arrays['profile_norm'] = (profile / scale).ravel()
    arrays['horizon'] = np.full(len(origins) * R, h, dtype=np.int8)
    arrays['dayofweek'] = np.repeat(dates.dayofweek, R).astype(np.int8)
    arrays['origin_minute'] = np.repeat(times[origins].hour * 60 + times[origins].minute, R).astype(np.int16)
    arrays['future_hour'] = np.repeat(dates.hour + dates.minute / 60, R).astype(np.float32)
    frame = pd.DataFrame(arrays)
    frame['cutoff'] = times[origins].repeat(R)
    frame['timestamp'] = dates.repeat(R)
    frame['route_id'] = np.tile(routes, len(origins)).astype(np.int16)
    frame['scale'] = scale.ravel()
    frame['profile28'] = profile.ravel()
    frame['y'] = past(target, slot).ravel()
    extra = {}
    for name in names[1:]:
        for lag in (4, 8, 12, 24, 48, 96):
            extra[f'long_{name}_lag{lag}'] = (past(values[name], origins - lag) / scale).ravel()
        for window, series in [(48, means[name, 48]), (336, long_week[name])]:
            extra[f'long_{name}_mean{window}'] = (series[origins] / scale).ravel()
    return extend_robust(pd.concat([frame, pd.DataFrame(extra)], axis=1))

In [ ]:
def prepare_history(branch):
    global values, target, means, level24, extra_signals, hub_signal, validation_lag_table, long_week, deviations
    values = branch
    target = values[TARGET]
    means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
    level24 = rolling(target, 48)
    positions = np.arange(len(times))
    extra_signals = {}
    for name in STATUSES[:3]:
        extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
    extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
    extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
    for name in STATUSES[3:]:
        extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
    extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
    extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
    extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
    hub_signal = extra_signals['status_6_diff_hourly']
    means.update({(name, 48): rolling(a, 48) for name, a in values.items()})
    long_week = {name: rolling(values[name], 336) for name in STATUSES}
    deviations = {w: pd.DataFrame(target).rolling(w, min_periods=2).std().to_numpy(dtype=np.float32) for w in (4, 12)}
    validation_lag_table = calculate_dynamic_shoulder(target, hub_signal, times, routes, fit_before=start_tune)

In [ ]:
cat_columns = ['route_id', 'target_back_0', 'target_back_1', 'target_back_2', 'target_back_4', 'target_1h_mean_4', 'target_1h_mean_12', 'status_1_mean_4', 'status_1_mean_12', 'status_1_latest', 'status_1_change_1h', 'status_2_mean_4', 'status_2_mean_12', 'status_2_latest', 'status_2_change_1h', 'status_3_mean_4', 'status_3_mean_12', 'status_3_latest', 'status_3_change_1h', 'status_4_mean_4', 'status_4_mean_12', 'status_4_latest', 'status_5_mean_4', 'status_5_mean_12', 'status_5_latest', 'status_6_mean_4', 'status_6_mean_12', 'status_6_latest', 'status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate', 'status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck', 'target_acceleration', 'target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48', 'target_vs_mean_6h', 'morning_level_ratio', 'hour', 'minute', 'hour_float', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available', 'slot_days_back_1', 'slot_days_back_2', 'slot_days_back_7', 'slot_daily7', 'slot_weekly4']
cat_trees = [1500, 1500, 1500, 1500, 1499, 1500, 1497, 1500]
anchor_columns = ['target_back_0', 'target_back_1', 'target_back_2', 'target_back_4', 'target_1h_mean_4', 'target_1h_mean_12', 'level24_feature', 'target_1h_lag_48', 'target_1h_lag_96', 'target_vs_mean_6h', 'morning_level_ratio', 'warehouse_processing_rate', 'target_acceleration']
anchor_columns += [f'status_{j}_{suffix}' for j in range(1, 7) for suffix in ('latest', 'mean_4', 'mean_12')]

def make_cat_frame(origins, h):
    return add_pilot_features(horizon_dataset(h, origins), h, origins)

def metric(actual, predicted):
    y = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    if y.shape != p.shape or not np.isfinite(y).all() or (not np.isfinite(p).all()) or (y.sum() <= 0):
        raise ValueError('Неверные данные для метрики')
    error = p - y
    wape = np.abs(error).sum() / y.sum()
    bias = error.sum() / y.sum()
    return dict(rows=len(y), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

In [ ]:
def residual_features(frame):
    cols = ['route_id', 'horizon', 'raw_blend', 'model_prediction', 'profile28', *anchor_columns]
    X = frame[cols].copy()
    X['dayofweek'] = frame.cutoff.dt.dayofweek
    X['route_id'] = X.route_id.astype(str)
    X['horizon'] = X.horizon.astype(str)
    for name in ['model_prediction', 'profile28', 'target_back_0', 'target_1h_mean_4', 'target_1h_mean_12', 'level24_feature', 'target_1h_lag_48', 'target_1h_lag_96']:
        X[f'{name}_minus_base'] = X[name] - X.raw_blend
    return X

In [ ]:
def segment_routes(boundary, stage):
    mask = TIMES < boundary
    y = VALUES[TARGET][mask].astype(np.float64)
    scale = np.maximum(y.mean(axis=0), 1.0)
    normalized = y / scale
    grid = TIMES[mask]
    descriptors = {}
    for slot in range(48):
        select = grid.hour * 2 + grid.minute // 30 == slot
        descriptors[f'slot_{slot:02}'] = normalized[select].mean(axis=0)
    for day in range(7):
        descriptors[f'weekday_{day}'] = normalized[grid.dayofweek == day].mean(axis=0)
    descriptors['log_cv'] = np.log1p(normalized.std(axis=0))
    descriptors['zero_share'] = (y == 0).mean(axis=0)
    descriptors['jump_share'] = (np.abs(np.diff(normalized, axis=0)) > 1).mean(axis=0)
    descriptors['log_q95'] = np.log1p(np.quantile(normalized, 0.95, axis=0))
    frame = pd.DataFrame(descriptors, index=ROUTES)
    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError('Недостаточно данных для профилей маршрутов')
    scaler = StandardScaler().fit(frame)
    z = scaler.transform(frame)
    labels, estimators = ({1: np.zeros(len(ROUTES), dtype=int)}, {})
    for k in CLUSTERS:
        if k == 1:
            continue
        model = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(z)
        groups = model.labels_.copy()
        counts = np.bincount(groups, minlength=k)
        groups[counts[groups] < MIN_CLUSTER_ROUTES] = -1
        labels[k], estimators[k] = (groups, model)
    artifact = dict(boundary=str(boundary), routes=ROUTES, scale=scale, labels=labels, descriptor_scaler=scaler, clusterers=estimators, descriptor_columns=list(frame.columns))
    joblib.dump(artifact, RUN / f'segments_{stage}.joblib')
    frame.insert(0, 'route_id', ROUTES)
    frame['target_scale'] = scale
    for k, groups in labels.items():
        frame[f'cluster_{k}'] = groups
    frame.to_csv(RUN / f'route_profiles_{stage}.csv', index=False)
    display(pd.DataFrame([dict(stage=stage, k=k, cluster=int(g), routes=int((groups == g).sum())) for k, groups in labels.items() for g in np.unique(groups)]))
    return artifact

In [ ]:
def normalize_shape(frame, shape, anchor):
    amounts = frame[group_keys + [shape, anchor]].copy()
    amounts[[shape, anchor]] = amounts[[shape, anchor]].astype(float)
    shape_total = amounts.groupby(group_keys, sort=False)[shape].transform('sum').to_numpy()
    anchor_total = amounts.groupby(group_keys, sort=False)[anchor].transform('sum').to_numpy()
    result = amounts[anchor].to_numpy().copy()
    valid = shape_total > 0
    result[valid] = amounts[shape].to_numpy()[valid] * anchor_total[valid] / shape_total[valid]
    totals = amounts.assign(prediction=result).groupby(group_keys, sort=False)[['prediction', anchor]].sum()
    if not np.allclose(totals.prediction, totals[anchor], rtol=1e-10, atol=1e-05):
        raise ValueError('Не сохранён общий объём')
    return result

In [ ]:
def make_volume_inputs(frame, h, model_prediction):
    result = frame[['cutoff', 'route_id', 'timestamp', 'profile28', *anchor_columns]].copy()
    result['horizon'] = h
    result['model_prediction'] = model_prediction
    result['raw_blend'] = model_prediction if h == 1 else .7 * model_prediction + .3 * result.profile28
    return result


def append_refit_metrics(rows, component, horizon, group, y, pred):
    row = dict(component=component, horizon=horizon, group=group, period='full_refit_in_sample', **metric(y, pred))
    rows.append(row)
    display(pd.DataFrame([row]))
    pd.DataFrame(rows).to_csv(RUN / 'full_refit_metrics.csv', index=False)


def train_full_bases(segmentation):
    route_groups = pd.Series(segmentation['labels'][8], index=routes)
    positions = np.arange(len(times))
    minutes = times.hour * 60 + times.minute
    future_origin = np.array([len(times) - 1])
    model_dir = RUN / 'models'
    model_dir.mkdir(exist_ok=True)
    outputs, rows = [], []
    for h in range(1, 9):
        eligible = positions[positions + h < len(times)]
        cat_columns_h = cat_columns + (SEASONAL_FEATURES if h in (5, 6, 8) else [])
        cat = CatBoostRegressor(iterations=cat_trees[h - 1], depth=7, learning_rate=.05, loss_function='MAE',
                                task_type='GPU', devices='0', cat_features=['route_id'], random_seed=42,
                                allow_writing_files=False, verbose=250, thread_count=4)
        train = pd.concat([make_cat_frame(eligible[start:start + 32], h)[cat_columns_h]
                           for start in range(0, len(eligible), 32)], ignore_index=True)
        cat.fit(train, target[eligible + h].ravel() / 100000)
        cat.save_model(str(model_dir / f'catboost_full_h{h}.cbm'))
        del train
        gc.collect()
        eval_origins = eligible[minutes[eligible] == 630]
        check = make_cat_frame(eval_origins, h)
        pred = np.maximum(cat.predict(check[cat_columns_h]) * 100000, 0)
        append_refit_metrics(rows, 'CatBoost', h, 'all', target[eval_origins + h].ravel(), pred)
        query = make_cat_frame(future_origin, h)
        result = make_volume_inputs(query, h, np.maximum(cat.predict(query[cat_columns_h]) * 100000, 0))
        del cat, check, query, pred
        gc.collect()
        lgb_origins = eligible[(eligible >= 336) & np.isin(minutes[eligible], [570, 600, 630, 660, 690])]
        train = session_shape_features(lgb_origins, h, target, level24)
        query = session_shape_features(future_origin, h, target, level24)
        if not result[keys].reset_index(drop=True).equals(query[keys].astype(result[keys].dtypes).reset_index(drop=True)):
            raise ValueError('Ключи CatBoost и LightGBM отличаются')
        config = meta[str(h)]
        columns = config['columns']
        if set(columns) & {'y', 'timestamp', 'cutoff'}:
            raise ValueError('Метка входит в признаки LightGBM')
        train_scale = train.scale.to_numpy(dtype=float)
        train_groups = train.route_id.map(route_groups).to_numpy()
        query_groups = query.route_id.map(route_groups).to_numpy()
        global_shape = np.empty(len(query))
        cluster_shape = np.empty(len(query))
        for group in ['global', *sorted(set(train_groups) - {-1})]:
            all_routes = group == 'global'
            fit = np.ones(len(train), dtype=bool) if all_routes else train_groups == group
            forecast = np.ones(len(query), dtype=bool) if all_routes else query_groups == group
            model = lgb.LGBMRegressor(**dict(config['params'], n_estimators=config['trees'], n_jobs=4))
            model.fit(train.loc[fit, columns], train.y.to_numpy()[fit] / train_scale[fit],
                      sample_weight=train_scale[fit] / train_scale.mean())
            model.booster_.save_model(str(model_dir / f'lightgbm_full_h{h}_{group}.txt'))
            check = fit & train.cutoff.dt.hour.eq(10).to_numpy() & train.cutoff.dt.minute.eq(30).to_numpy()
            pred = np.maximum(model.predict(train.loc[check, columns]) * train_scale[check], 0)
            append_refit_metrics(rows, 'LightGBM', h, group, train.y.to_numpy()[check], pred)
            shape = np.maximum(model.predict(query.loc[forecast, columns]) * query.scale.to_numpy()[forecast], 0)
            if all_routes:
                global_shape[:] = shape
                cluster_shape[:] = shape
            else:
                cluster_shape[forecast] = shape
            del model, pred, shape
        outputs.append(result.assign(global_shape=global_shape, clustered_shape=cluster_shape))
        del train, query
        gc.collect()
    return pd.concat(outputs, ignore_index=True), rows


def historical_residual_inputs():
    history = pd.read_parquet(RESIDUAL_INPUTS['selected_evaluation.parquet'])
    anchors = pd.read_parquet(RESIDUAL_INPUTS['anchor_features.parquet'], columns=['cutoff', 'route_id', *anchor_columns])
    history = history.merge(anchors, on=['cutoff', 'route_id'], how='left', validate='many_to_one')
    source = times.get_indexer(pd.DatetimeIndex(history.timestamp))
    indices = pd.Index(routes).get_indexer(history.route_id)
    if history.duplicated(keys).any() or history[anchor_columns].isna().any().any() or (source < 0).any() or (indices < 0).any():
        raise ValueError('Некорректные исторические OOF-прогнозы')
    if not history.timestamp.eq(history.cutoff + pd.to_timedelta(history.horizon * 30, unit='m')).all() or not np.array_equal(history.y.to_numpy(), target[source, indices]):
        raise ValueError('OOF-факты или время не совпадают с историей')
    if len(history) != 28 * 8 * len(routes) or history.timestamp.max() >= times[-1] - pd.Timedelta(hours=4):
        raise ValueError('Неожиданный период OOF для моделей поправки')
    return history


def refit_volume(history, future, rows):
    fit = residual_features(history)
    query = residual_features(future)
    known = history.timestamp < times[-1] - pd.Timedelta(hours=4)
    age = (times[-1] - history.loc[known, 'cutoff']).dt.total_seconds().to_numpy() / 86400
    target_residual = (history.y.to_numpy() - history.raw_blend.to_numpy()) / 100000
    forecasts, audit = [], []
    for depth, trees, l2 in [(4, 450, 30), (5, 900, 50)]:
        model = CatBoostRegressor(iterations=trees, depth=depth, learning_rate=.03, l2_leaf_reg=l2,
                                  loss_function='MAE', cat_features=['route_id', 'horizon'], random_seed=42,
                                  task_type='GPU', devices='0', thread_count=4, allow_writing_files=False, verbose=False)
        model.fit(fit, target_residual)
        model.save_model(str(RUN / 'models' / f'residual_full_depth{depth}.cbm'))
        in_sample = history.raw_blend.to_numpy() + model.predict(fit) * 100000
        errors = history.y.to_numpy() - in_sample
        offset = float(np.average(errors[known], weights=np.exp2(-age / 7)))
        append_refit_metrics(rows, f'Residual depth {depth}', 'all', 'all', history.y.to_numpy(), np.maximum(in_sample, 0))
        forecasts.append(np.maximum(future.raw_blend.to_numpy() + model.predict(query) * 100000 + offset, 0))
        audit.append(dict(depth=depth, trees=trees, ewma_offset=offset, fit_through=str(history.timestamp.max()),
                          bias_fit_through=str(history.loc[known, 'timestamp'].max())))
    pd.DataFrame(audit).to_csv(RUN / 'volume_refit_audit.csv', index=False)
    return np.mean(forecasts, axis=0)


def write_submission(requests, forecast, prediction, filename):
    output = requests.merge(forecast[['route_id', 'timestamp']].assign(y_pred=prediction), on=['route_id', 'timestamp'],
                            how='left', sort=False, validate='one_to_one')
    if len(output) != 8000 or not output.id.is_unique or not output.id.equals(requests.id) or not np.isfinite(output.y_pred).all() or output.y_pred.lt(0).any():
        raise ValueError('Некорректный submission')
    output[['id', 'y_pred']].to_csv(RUN / filename, index=False)


def run_full_submissions():
    global start_tune
    start_tune = times[-1] + STEP
    historical_scores = pd.DataFrame(REFERENCE_SCORES)
    historical_scores.to_csv(RUN / 'test_before_full_refit.csv', index=False)
    print('Архивная независимая от полного refit оценка test 18–31 октября. Этот период ранее изучался.')
    display(historical_scores)
    if get_gpu_device_count() < 1:
        raise RuntimeError('Включите GPU для CatBoost')
    requests = pd.read_parquet(TEST_INPUT, columns=['id', 'route_id', 'timestamp'])
    requests['timestamp'] = pd.to_datetime(requests.timestamp)
    history = historical_residual_inputs()
    with threadpool_limits(limits=4):
        prepare_history(values)
        segmentation = segment_routes(start_tune, 'full')
        future, rows = train_full_bases(segmentation)
        future['volume_model'] = refit_volume(history, future, rows)
    weight = float(meta['weight'])
    for shape, filename in [('global_shape', 'submission.csv'), ('clustered_shape', 'submission_clusters.csv')]:
        adjusted = normalize_shape(future, shape, 'volume_model')
        pred = (1 - weight) * future.volume_model.to_numpy() + weight * adjusted
        sums = future.assign(pred=pred).groupby(group_keys)[['pred', 'volume_model']].sum()
        if not np.allclose(sums.pred, sums.volume_model, rtol=1e-10, atol=1e-5):
            raise ValueError('Нарушено сохранение объёма горизонта')
        future[shape + '_calibrated'] = pred
        write_submission(requests, future, pred, filename)
    future.to_parquet(RUN / 'future_predictions.parquet', index=False)
    (RUN / 'protocol.json').write_text(json.dumps(dict(fit_end_exclusive=str(start_tune),
        shape_weight=weight, selected_shape='global', local_test_evaluation_before_full_refit=True,
        volume_catboost_starts='all_eligible', shape_lightgbm_starts=[570, 600, 630, 660, 690],
        residual_fit_scope='archived_raw_oof_oct04_oct31', new_local_test_score=False), ensure_ascii=False, indent=2))
    with zipfile.ZipFile(RUN.with_suffix('.zip'), 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=2) as archive:
        for path in sorted(RUN.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(RUN.parent))
    print('Готовые файлы', RUN / 'submission.csv', RUN / 'submission_clusters.csv')
    print('ZIP', RUN.with_suffix('.zip'))


In [ ]:
run_full_submissions()
